In [1]:
import os
import requests
from typing import Optional, List
from guam.datalibs import DataLoader
from omegaconf import OmegaConf


PATH = "./datasets/"
DATA_PATH = os.path.join(PATH, "titanic.csv")
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

try:
    print("Download Titanic Datasets...")
    response = requests.get(url)
    response.raise_for_status()  # 요청이 실패하면 에러를 발생시킴

    with open(DATA_PATH, 'wb') as file:
        file.write(response.content)

    print(f"Download Succses")
    print(f"Save Path: {PATH}")

except requests.exceptions.RequestException as e:
    print(f"Error Occured!!: {e}")




Download Titanic Datasets...
Download Succses
Save Path: ./datasets/


In [2]:
# Make Workflow config
yaml_string = """
    data:
      output_path: "./final_processed_data"

    workflow:
        - selection: ["PassengerId","Survived","Pclass","Name","SibSp","Parch","Ticket","Cabin"]
        - selection: ["Age", "Fare"]
          ops:
            - FillMissing:
                fill_val: 0
            - LogOp:
        
        - selection: ["Age"]
          ops:
            - Normalize:
          output_name: ["age_normalized"]

        - selection: ["Sex", "Embarked"]
          ops:
            - Categorify:
                freq_threshold: 10
    """
    
# 2. Load the configuration using OmegaConf
cfg = OmegaConf.create(yaml_string)

In [3]:
cfg.workflow

[{'selection': ['PassengerId', 'Survived', 'Pclass', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin']}, {'selection': ['Age', 'Fare'], 'ops': [{'FillMissing': {'fill_val': 0}}, {'LogOp': None}]}, {'selection': ['Age'], 'ops': [{'Normalize': None}], 'output_name': ['age_normalized']}, {'selection': ['Sex', 'Embarked'], 'ops': [{'Categorify': {'freq_threshold': 10}}]}]

In [4]:
TestDataLoader = DataLoader(DATA_PATH)
workflow = TestDataLoader._build_workflow(cfg.workflow)

/home/cvmi_jeyoung/anaconda3/envs/rapids-nvt/lib/python3.10/site-packages/dask_cudf/io/csv.py:79: FutureWarning: `chunksize` is deprecated and will be removed in the future. Please use `blocksize` instead.
  warn(


In [5]:
TestDataLoader(cfg.workflow)

Fitting and transforming dataset with the workflow...


/home/cvmi_jeyoung/anaconda3/envs/rapids-nvt/lib/python3.10/site-packages/dask_cudf/io/csv.py:79: FutureWarning: `chunksize` is deprecated and will be removed in the future. Please use `blocksize` instead.
  warn(


Saving processed data to ./processed_data...
Processing complete.


/home/cvmi_jeyoung/anaconda3/envs/rapids-nvt/lib/python3.10/site-packages/dask_cudf/io/csv.py:79: FutureWarning: `chunksize` is deprecated and will be removed in the future. Please use `blocksize` instead.
  warn(
/home/cvmi_jeyoung/anaconda3/envs/rapids-nvt/lib/python3.10/site-packages/dask_cudf/io/csv.py:79: FutureWarning: `chunksize` is deprecated and will be removed in the future. Please use `blocksize` instead.
  warn(


(<nvtabular.workflow.workflow.Workflow at 0x7f94ca1219c0>, './processed_data')

In [6]:
import pandas as pd

df = pd.read_parquet(os.path.join(".", "processed_data", "part_0.parquet"))
df

,PassengerId,Survived,Pclass,Name,SibSp,Parch,Ticket,Cabin,Age,Fare,['age_normalized'],Sex,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",1,0,A/5 21171,<NA>,3.135494,2.110213,-0.530005,3,3
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,PC 17599,C85,3.663562,4.280593,0.57143,4,4
2,3,1,3,"Heikkinen, Miss. Laina",0,0,STON/O2. 3101282,<NA>,3.295837,2.188856,-0.254646,4,3
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,113803,C123,3.583519,3.990834,0.364911,4,3
4,5,0,3,"Allen, Mr. William Henry",0,0,373450,<NA>,3.583519,2.202765,0.364911,3,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",0,0,211536,<NA>,3.332205,2.639057,-0.185807,3,3
887,888,1,1,"Graham, Miss. Margaret Edith",0,0,112053,B42,2.995732,3.433987,-0.736524,4,3
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",1,2,W./C. 6607,<NA>,0.000000,3.196630,<NA>,4,3
889,890,1,1,"Behr, Mr. Karl Howell",0,0,111369,C148,3.295837,3.433987,-0.254646,3,4
